# ECMWF AIFS-ENS — Mean/Spread Analysis

**Model:** ECMWF AIFS Ensemble — 51 members
**Station:** TGPY, Grenada

In [ ]:
import cfgrib, numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

TGPY_LAT, TGPY_LON = 12.0042, -61.7862
UTC_OFFSET = pd.Timedelta(hours=-4)
RUN_DIR = Path("data") / sorted(Path("data").iterdir(), reverse=True)[0].name

def nearest_point(ds):
    return ds.sel(latitude=TGPY_LAT, longitude=TGPY_LON, method="nearest", tolerance=0.5)

def grib_to_df(filepath):
    datasets = cfgrib.open_datasets(str(filepath))
    series = {}
    for ds in datasets:
        pt = nearest_point(ds)
        for var in pt.data_vars:
            da = pt[var]
            vt = da.valid_time.values
            idx = pd.to_datetime([vt]) if vt.ndim == 0 else pd.to_datetime(vt)
            val = [float(da.values)] if vt.ndim == 0 else da.values.astype(float)
            series[var] = pd.Series(val, index=idx)
    return pd.DataFrame(series).sort_index()

print(f"Run: {RUN_DIR.name}")

In [ ]:
df_mean = grib_to_df(RUN_DIR / "aifs_ens_mean_f000-f120_6h.grib2")
df_sprd = grib_to_df(RUN_DIR / "aifs_ens_spread_f000-f120_6h.grib2")

run_init = df_mean.index[0]
t2m_col  = "t2m" if "t2m" in df_mean.columns else None
msl_col  = "msl" if "msl" in df_mean.columns else None
si_col   = next((c for c in df_mean.columns if "si" in c or "speed" in c.lower()), None)

print("ECMWF AIFS-ENS — Mean/Spread at TGPY (0-120h)")
print("-" * 60)
print(f"  {'Step':>5}  {'AST':>14}  {'T2m mean':>9}  {'T2m sprd':>9}  {'MSL mean':>9}")
print("-" * 60)
for ts in df_mean.index:
    step_h = int((pd.Timestamp(ts) - pd.Timestamp(run_init)).total_seconds() / 3600)
    ast    = (pd.Timestamp(ts) + UTC_OFFSET).strftime("%d%b %H:%M")
    t_m = (float(df_mean[t2m_col][ts]) - 273.15) if t2m_col else float("nan")
    t_s =  float(df_sprd[t2m_col][ts])            if t2m_col and t2m_col in df_sprd.columns else float("nan")
    m_m = (float(df_mean[msl_col][ts]) / 100)     if msl_col else float("nan")
    print(f"  {step_h:>5}  {ast:>14}  {t_m:>9.1f}  {t_s:>9.2f}  {m_m:>9.1f}")

In [ ]:
# ── Canonical derivation + export (from AIFS-ENS mean) ────────────────────────
CANONICAL_COLS = [
    # Core (original 11)
    "t2m_c", "msl_hpa", "wspd_kt", "wdir", "d2m_c", "tcc_pct",
    "tp_mm", "tcwv", "fg10_kt", "sp_hpa", "cape",
    # New tropical params (most are NaN for AIFS-ENS mean — only 3 params available)
    "skt_c", "cp_mm", "lcc_pct", "mcc_pct", "hcc_pct",
    "cin", "olr_wm2", "blh_m", "lhf", "shf",
]

am = df_mean.copy()
if "t2m" in am.columns: am["t2m_c"]   = am["t2m"] - 273.15
if "msl" in am.columns: am["msl_hpa"] = am["msl"] / 100

# AIFS-ENS mean provides 10si (scalar wind speed, m/s) — no u/v components
si_col = next((c for c in am.columns if c in ("si10", "10si", "ws10") or
               ("si" in c and len(c) <= 5)), None)
if si_col:
    am["wspd_kt"] = am[si_col] * 1.944
# All other canonical cols will be NaN — that is correct for the mean product

export = pd.DataFrame(
    {col: am[col].values if col in am.columns else [float("nan")] * len(am)
     for col in CANONICAL_COLS},
    index=am.index,
)
export.insert(0, "model", "ecmwf_aifs_ens")
export.index.name = "valid_time"

csv_path = RUN_DIR / "canonical_sfc.csv"
export.to_csv(csv_path, float_format="%.4f")
non_nan = export.notna().sum().sum()
print(f"Saved: {csv_path.name}  ({len(export)} steps, {non_nan} non-NaN values)")
print(export[["t2m_c", "msl_hpa", "wspd_kt"]].round(2).to_string())